# End-to-End RAG Pipeline Demo

This notebook demonstrates the complete RAG (Retrieval-Augmented Generation) pipeline.

**Pipeline**: Query → Embedding → Search → Retrieval → Augment → LLM → Answer

**Goal**: Show the full workflow from question to answer with sources

In [ ]:
import sys
sys.path.insert(0, '../src')

from retrieval import RetrievalService
from generation import GenerationService
from ingestion import DataIngestionService
import json

print("Modules imported successfully")

## 1. Initialize Pipeline

In [ ]:
retrieval_service = RetrievalService()
generation_service = GenerationService()
ingestion_service = DataIngestionService()

doc_count = ingestion_service.get_document_count()
print(f"Vector Database Status:")
print(f"  Documents available: {doc_count}")
print(f"  Embedding model: text-embedding-3-large")
print(f"  LLM model: gpt-4o")

if doc_count == 0:
    print("\n⚠️  No documents loaded. Run notebook 01_data_inspection first to load data.")
else:
    print("\n✓ All components ready")

## 2. Simple RAG Query

In [ ]:
# Example 1: Simple question
question = "What is artificial intelligence?"

try:
    # Step 1: Retrieve relevant documents
    print(f"Question: {question}\n")
    print("Step 1: Retrieving relevant documents...")
    documents = await retrieval_service.retrieve_documents(query=question, top_k=3)
    print(f"Retrieved {len(documents)} documents")
    
    for i, doc in enumerate(documents, 1):
        print(f"\n  Doc {i} (relevance: {doc['score']:.4f}):")
        print(f"  {doc['content'][:150]}...")
    
    # Step 2: Generate answer using retrieved documents
    print(f"\nStep 2: Generating answer using LLM...")
    result = await generation_service.generate_answer(
        query=question,
        documents=documents,
        temperature=0.7
    )
    
    print(f"\nAnswer:")
    print(f"{result['answer']}")
    
except Exception as e:
    print(f"Error in RAG pipeline: {e}")

## 3. Step-by-Step Pipeline Breakdown

In [ ]:
# Show each step of the pipeline in detail
question = "What are neural networks?"

try:
    print(f"Question: {question}\n")
    print("=" * 70)
    
    # STEP 1: Create embedding for question
    print("\n[STEP 1] QUERY EMBEDDING")
    print("-" * 70)
    embedding = retrieval_service.get_embedding(question)
    print(f"Embedding created for query")
    print(f"  Dimension: {len(embedding)}")
    print(f"  Sample values: {embedding[:5]}")
    
    # STEP 2: Search vector database
    print("\n[STEP 2] VECTOR DATABASE SEARCH")
    print("-" * 70)
    documents = await retrieval_service.retrieve_documents(query=question, top_k=3)
    print(f"Top-k documents retrieved: {len(documents)}")
    
    # STEP 3: Examine retrieved documents
    print("\n[STEP 3] RETRIEVED DOCUMENTS")
    print("-" * 70)
    total_chars = 0
    for i, doc in enumerate(documents, 1):
        chars = len(doc['content'])
        total_chars += chars
        print(f"\nDocument {i}:")
        print(f"  Relevance Score: {doc['score']:.4f}")
        print(f"  Size: {chars} characters")
        print(f"  Content preview: {doc['content'][:100]}...")
    
    print(f"\nTotal context: {total_chars} characters")
    
    # STEP 4: Augment prompt
    print("\n[STEP 4] AUGMENTED PROMPT")
    print("-" * 70)
    augmented_prompt = generation_service._create_augmented_prompt(question, documents)
    print(f"Prompt created")
    print(f"  Total prompt length: {len(augmented_prompt)} characters")
    print(f"  First 200 chars:\n{augmented_prompt[:200]}...")
    
    # STEP 5: Generate answer
    print("\n[STEP 5] LLM INFERENCE")
    print("-" * 70)
    result = await generation_service.generate_answer(
        query=question,
        documents=documents,
        temperature=0.7
    )
    print(f"Answer generated by GPT-4o")
    
    # STEP 6: Display final answer
    print("\n[STEP 6] FINAL ANSWER")
    print("=" * 70)
    print(f"\n{result['answer']}")
    print("\n" + "=" * 70)
    
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

## 4. Multiple Questions Test

In [ ]:
# Test the system with multiple questions
test_questions = [
    "What is machine learning?",
    "How do neural networks work?",
    "What is deep learning?",
]

for idx, question in enumerate(test_questions, 1):
    try:
        print(f"\n{'='*70}")
        print(f"Question {idx}: {question}")
        print('='*70)
        
        # Retrieve documents
        documents = await retrieval_service.retrieve_documents(query=question, top_k=2)
        
        # Generate answer
        result = await generation_service.generate_answer(
            query=question,
            documents=documents,
            temperature=0.7
        )
        
        print(f"\nAnswer:\n{result['answer'][:500]}...")
        print(f"\nSources: {len(documents)} documents retrieved")
        
    except Exception as e:
        print(f"Error processing question: {e}")

## 5. Answer Quality Analysis

In [ ]:
# Analyze the quality of generated answers
question = "What is artificial intelligence?"

try:
    # Retrieve documents
    documents = await retrieval_service.retrieve_documents(query=question, top_k=3)
    
    # Generate answer
    result = await generation_service.generate_answer(
        query=question,
        documents=documents,
        temperature=0.7
    )
    
    # Analyze quality
    metrics = generation_service.validate_answer_quality(result['answer'], documents)
    
    print(f"Answer Quality Metrics:")
    print(f"\nQuestion: {question}")
    print(f"\nAnswer length: {metrics['answer_length']} characters")
    print(f"Answer word count: {metrics['answer_words']} words")
    print(f"Has citations: {metrics['has_citations']}")
    print(f"Source documents: {metrics['document_count']}")
    print(f"Average source relevance: {metrics['avg_document_score']:.4f}")
    
    # Quality assessment
    print(f"\nQuality Assessment:")
    if metrics['answer_length'] > 300:
        print("  ✓ Adequate answer length")
    else:
        print("  ⚠ Answer could be more detailed")
    
    if metrics['has_citations']:
        print("  ✓ Answer includes citations")
    else:
        print("  ⚠ Answer lacks source citations")
    
    if metrics['avg_document_score'] > 0.6:
        print("  ✓ High quality source documents")
    else:
        print("  ⚠ Source document relevance could be higher")
    
except Exception as e:
    print(f"Error: {e}")

## 6. Temperature Impact on Answers

In [ ]:
# Show how temperature affects answer generation
question = "What are some applications of machine learning?"
temperatures = [0.0, 0.5, 1.0]

try:
    documents = await retrieval_service.retrieve_documents(query=question, top_k=2)
    
    print(f"Question: {question}\n")
    print(f"Retrieved {len(documents)} context documents\n")
    
    for temp in temperatures:
        print(f"\n{'='*70}")
        print(f"Temperature: {temp}")
        print(f"(0.0=deterministic, 1.0=creative)")
        print('='*70)
        
        result = await generation_service.generate_answer(
            query=question,
            documents=documents,
            temperature=temp
        )
        
        print(f"\n{result['answer'][:400]}...\n")
        
except Exception as e:
    print(f"Error: {e}")

## 7. RAG System Summary

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════╗
║            RAG SYSTEM WORKFLOW SUMMARY                            ║
╚════════════════════════════════════════════════════════════════════╝

1. USER QUESTION
   └─ Input natural language question

2. QUERY EMBEDDING
   └─ Convert question to vector using text-embedding-3-large
   └─ Enables semantic similarity matching

3. VECTOR SEARCH
   └─ Find most similar documents in database
   └─ Ranked by cosine similarity score
   └─ Retrieve top-k results (typically 3-10)

4. DOCUMENT RETRIEVAL
   └─ Extract full text and metadata from found documents
   └─ Provides relevant context for LLM

5. PROMPT AUGMENTATION  
   └─ Insert retrieved documents into LLM prompt
   └─ Includes relevance scores and document citations
   └─ Provides grounding for answer generation

6. LLM INFERENCE
   └─ Send augmented prompt to GPT-4o
   └─ Model generates answer based on context
   └─ Temperature controls response creativity

7. ANSWER DELIVERY
   └─ Return generated answer with source documents
   └─ User can verify claims against sources
   └─ Enables explainability and transparency

╔════════════════════════════════════════════════════════════════════╗
║           KEY ADVANTAGES OF RAG APPROACH                          ║
╚════════════════════════════════════════════════════════════════════╝

✓ Grounded Answers    - Responses based on actual data
✓ Explainable         - Sources can be cited and verified
✓ Up-to-Date         - Uses current data in knowledge base
✓ Reduced Hallucination - LLM constrained by context
✓ Transparent         - Clear traceability of information flow
✓ Cost-Effective      - Uses smaller prompts and models
✓ Traceable           - Full pipeline is observable
""")